# Exercises XP — Minimal MCP over STDIO

## Complete, documented solution

This notebook creates a tiny **Model Context Protocol (MCP)** project with:

- an MCP server named `Demo`;
- one callable tool: `add(a, b)`;
- one read-only resource template: `greeting://{name}`;
- a Python client that starts the server over STDIO;
- discovery, resource reading, tool invocation, and smoke tests.

The notebook writes the two files required for submission:

- `server.py`
- `client.py`

## Learning objectives

You will learn:

1. the difference between an MCP host, client, and server;
2. why STDIO is convenient for a local child process;
3. how tools differ from resources;
4. how Python type hints become MCP schemas;
5. how the client/server initialization lifecycle works;
6. how to discover capabilities before using them;
7. how to parse MCP resource and tool results;
8. how to troubleshoot a closed STDIO connection.

# 1. MCP architecture

## Host

The host is the user-facing application that wants access to external
context or actions. A desktop AI application, IDE, or custom agent may act
as a host.

## Client

The MCP client lives inside or beside the host. It opens a connection to one
server, negotiates capabilities, sends requests, and parses responses.

## Server

The MCP server exposes controlled capabilities:

- **Resources:** read-only context, comparable to file-like data or a GET.
- **Tools:** callable actions, comparable to a function or a POST.
- **Prompts:** reusable interaction templates.

In this exercise, our Python program is both the demonstration host and MCP
client, while `server.py` is the server.

## Why STDIO is ideal for local development

```text
client.py
   │
   ├── starts: mcp run server.py
   │
   ├── writes MCP JSON-RPC to the server's stdin
   │
   └── reads MCP JSON-RPC from the server's stdout
```

Benefits:

- no port selection;
- no HTTP server;
- the server can live only as long as the client;
- local permissions remain controlled by the parent process;
- setup is small and easy to debug.

**Critical rule:** a STDIO server must not print normal logs to stdout,
because stdout carries protocol messages. Use stderr or a logging system.

# 2. Install the stable MCP Python SDK

In [ ]:
# The official SDK currently has a stable v1 line and a pre-release v2 line.
# This exercise uses the v1 FastMCP and ClientSession APIs.
#
# `<2` prevents a future major upgrade from silently breaking the notebook.

%pip install -qU "mcp[cli]>=1.27,<2"

In [ ]:
# Verify the interpreter and installed MCP package.

import importlib.metadata as metadata
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("MCP SDK version:", metadata.version("mcp"))

major, minor = sys.version_info[:2]

assert (major, minor) >= (3, 10), (
    "MCP requires Python 3.10 or newer."
)

In [ ]:
# Verify that the MCP command-line application is visible.
#
# In a terminal, an unavailable command usually means that the virtual
# environment is not activated.

!mcp --help | head -n 12

# 3. Create `server.py`

The server uses `FastMCP`, which reads:

- the function name;
- Python type hints;
- the function docstring.

It automatically generates the MCP tool/resource schemas and handles the
protocol requests.

In [ ]:
%%writefile server.py
"""Minimal MCP server exposed over STDIO.

The server provides:
- one tool: add(a, b)
- one resource template: greeting://{name}

Important STDIO rule:
Do not use print() for normal logging in this server. Standard output is
reserved for MCP JSON-RPC messages. Log to stderr or a file instead.
"""

from mcp.server.fastmcp import FastMCP


# FastMCP converts Python type hints and docstrings into MCP schemas.
# The server name is visible to clients during session initialization.
mcp = FastMCP("Demo")


@mcp.tool()
def add(a: int, b: int) -> int:
    """Return the sum of two integers.

    Args:
        a: First integer.
        b: Second integer.

    Returns:
        The integer sum a + b.
    """
    return a + b


@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Return a personalized greeting resource.

    The URI is a resource template. A client can replace {name} with a
    concrete value, for example greeting://hello.

    Args:
        name: Value captured from the resource URI.

    Returns:
        A short greeting.
    """
    return f"Hello, {name}!"


def main() -> None:
    """Start the MCP server using the STDIO transport.

    STDIO is ideal for local development because the client starts the
    server as a child process and communicates through stdin/stdout.
    No TCP port, web server, or authentication setup is required.
    """
    mcp.run(transport="stdio")


if __name__ == "__main__":
    main()

## Server walkthrough

### `FastMCP("Demo")`

Creates the server and gives it a human-readable identity.

### `@mcp.tool()`

Registers `add` as an action. The annotations `a: int`, `b: int`, and
`-> int` define its input/output contract.

### `@mcp.resource("greeting://{name}")`

Registers a dynamic read-only resource. `{name}` is extracted from the URI.

### `mcp.run(transport="stdio")`

Starts the protocol loop using standard input and standard output.

In [ ]:
# Display the generated file with line numbers for easier study.

from pathlib import Path

for line_number, line in enumerate(
    Path("server.py").read_text(encoding="utf-8").splitlines(),
    start=1,
):
    print(f"{line_number:>3}: {line}")

In [ ]:
# Compile the file without starting the STDIO server.
# A SyntaxError here would stop before the client is created.

import py_compile

py_compile.compile(
    "server.py",
    doraise=True,
)

print("server.py syntax: OK")

# 4. Create `client.py`

The client performs the normal MCP lifecycle:

```text
start child process
      ↓
create ClientSession
      ↓
initialize
      ↓
list capabilities
      ↓
read resource
      ↓
call tool
      ↓
close session and child process
```

In [ ]:
%%writefile client.py
"""Minimal MCP client that connects to server.py over STDIO.

The client:
1. starts the server through the MCP CLI;
2. initializes an MCP session;
3. discovers resources, resource templates, and tools;
4. reads greeting://hello;
5. calls add(a=1, b=7).
"""

import asyncio
import os
import shutil
from pathlib import Path
from typing import Any

from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client
from pydantic import AnyUrl


# Resolve server.py relative to this client file.
# This makes the client work even when launched from another directory.
SERVER_PATH = Path(__file__).resolve().with_name("server.py")


def find_mcp_command() -> str:
    """Return the MCP CLI path or raise a useful setup error."""
    command = shutil.which("mcp")

    if command is None:
        raise RuntimeError(
            "The 'mcp' command was not found. Activate the virtual "
            "environment and install 'mcp[cli]>=1.27,<2'."
        )

    return command


# StdioServerParameters describes the child process that the client starts.
#
# Equivalent terminal command:
#     mcp run /absolute/path/to/server.py
#
# os.environ.copy() forwards the current environment to the child process.
server_params = StdioServerParameters(
    command=find_mcp_command(),
    args=["run", str(SERVER_PATH)],
    env=os.environ.copy(),
)


def resource_template_uri(template: Any) -> str:
    """Read a resource-template URI across compatible SDK field names."""
    value = getattr(template, "uriTemplate", None)

    if value is None:
        value = getattr(template, "uri_template", None)

    return str(value)


def extract_resource_text(result: Any) -> str:
    """Extract the first text block from a ReadResourceResult."""
    for block in getattr(result, "contents", []):
        text = getattr(block, "text", None)

        if text is not None:
            return str(text)

    return str(result)


def extract_tool_value(result: Any) -> Any:
    """Extract a structured or text value from a CallToolResult.

    FastMCP commonly returns:
    - structuredContent = {"result": 8}
    - content = [TextContent(text="8")]

    Prefer structured data, then fall back to text.
    """
    structured = getattr(result, "structuredContent", None)

    if structured is None:
        structured = getattr(result, "structured_content", None)

    if isinstance(structured, dict):
        if "result" in structured:
            return structured["result"]

        return structured

    for block in getattr(result, "content", []):
        if isinstance(block, types.TextContent):
            return block.text

        text = getattr(block, "text", None)
        if text is not None:
            return text

    return str(result)


async def run() -> None:
    """Connect to the server, discover capabilities, and invoke them."""

    # stdio_client starts the server child process and exposes two streams:
    # - read_stream: messages coming from the server;
    # - write_stream: messages sent to the server.
    async with stdio_client(server_params) as (
        read_stream,
        write_stream,
    ):
        # ClientSession implements the MCP lifecycle and request methods.
        async with ClientSession(
            read_stream,
            write_stream,
        ) as session:
            # The client and server exchange protocol versions and
            # capabilities before any normal request is made.
            initialization = await session.initialize()

            print(
                "Connected server:",
                initialization.serverInfo.name,
            )

            # list_resources() returns concrete/static resources.
            resources_result = await session.list_resources()
            resource_uris = [
                str(resource.uri)
                for resource in resources_result.resources
            ]

            # greeting://{name} is dynamic, so it appears as a template.
            templates_result = (
                await session.list_resource_templates()
            )
            template_uris = [
                resource_template_uri(template)
                for template
                in templates_result.resourceTemplates
            ]

            # list_tools() returns tools and their JSON schemas.
            tools_result = await session.list_tools()
            tool_names = [
                tool.name
                for tool in tools_result.tools
            ]

            print("Static resources:", resource_uris)
            print("Resource templates:", template_uris)
            print("Tools:", tool_names)

            # Instantiate the template with name="hello".
            greeting_result = await session.read_resource(
                AnyUrl("greeting://hello")
            )
            greeting_text = extract_resource_text(
                greeting_result
            )

            print(
                "greeting://hello ->",
                greeting_text,
            )

            # Tool arguments are sent as a JSON-compatible dictionary.
            addition_result = await session.call_tool(
                "add",
                arguments={
                    "a": 1,
                    "b": 7,
                },
            )
            addition_value = extract_tool_value(
                addition_result
            )

            print("add(1, 7) ->", addition_value)

            # Simple assertions turn this demo into a smoke test.
            assert "greeting://{name}" in template_uris
            assert "add" in tool_names
            assert greeting_text == "Hello, hello!"
            assert str(addition_value) == "8"


def main() -> None:
    """Synchronous entry point for the asynchronous client."""
    asyncio.run(run())


if __name__ == "__main__":
    main()

## Important client details

### `StdioServerParameters`

Describes the command that starts the child server. The notebook follows
the assignment and uses the MCP CLI:

```text
mcp run server.py
```

### `stdio_client`

Starts the process and yields asynchronous read/write streams.

### `ClientSession`

Implements initialization, discovery, resource requests, and tool calls.

### Resources versus resource templates

`greeting://{name}` is not one fixed resource. It is a template that can
produce many resources, such as:

- `greeting://hello`
- `greeting://Fahim`
- `greeting://student`

It therefore appears in `list_resource_templates()`. The notebook still
calls `list_resources()` to show the distinction.

In [ ]:
# Display the client with line numbers.

for line_number, line in enumerate(
    Path("client.py").read_text(encoding="utf-8").splitlines(),
    start=1,
):
    print(f"{line_number:>3}: {line}")

In [ ]:
# Static syntax validation for both deliverable files.

for filename in ["server.py", "client.py"]:
    py_compile.compile(
        filename,
        doraise=True,
    )
    print(f"{filename} syntax: OK")

# 5. Run the complete STDIO integration

In [ ]:
# Use subprocess instead of a notebook shell command so that:
# - stdout can be saved as the required terminal capture;
# - stderr can be displayed separately;
# - timeout and return code can be checked.

import subprocess

completed = subprocess.run(
    [sys.executable, "client.py"],
    capture_output=True,
    text=True,
    timeout=30,
    check=False,
)

print("CLIENT STDOUT")
print(completed.stdout)

if completed.stderr.strip():
    print("SERVER/CLIENT STDERR")
    print(completed.stderr)

print("Return code:", completed.returncode)

assert completed.returncode == 0, (
    "The integration failed. Read stderr above."
)

## Expected output

The exact log formatting can vary by MCP version, but the client output
should include:

```text
Connected server: Demo
Static resources: []
Resource templates: ['greeting://{name}']
Tools: ['add']
greeting://hello -> Hello, hello!
add(1, 7) -> 8
```

An empty static resource list is correct: our greeting is a **template**.

In [ ]:
# Save the submission-friendly text capture.

capture_text = (
    "$ python client.py\n"
    + completed.stdout
)

Path("terminal_capture.txt").write_text(
    capture_text,
    encoding="utf-8",
)

print(capture_text)
print("Saved: terminal_capture.txt")

# 6. Optional direct function checks

These checks do not use MCP transport. They simply verify the underlying
Python logic before protocol-level testing.

In [ ]:
import importlib.util

specification = importlib.util.spec_from_file_location(
    "demo_server",
    Path("server.py").resolve(),
)
demo_server = importlib.util.module_from_spec(specification)
specification.loader.exec_module(demo_server)

assert demo_server.add(1, 7) == 8
assert demo_server.greet("hello") == "Hello, hello!"

print("Direct add check: 8")
print("Direct greeting check: Hello, hello!")

# 7. Run from local terminals

## Recommended one-terminal mode

The client starts and stops the server:

```bash
python client.py
```

## Manual server inspection

Terminal 1:

```bash
mcp run server.py
```

The server waits for MCP protocol input; it is not an ordinary interactive
prompt.

For normal exercise execution, use `python client.py`. The client manages
the child process and speaks the protocol correctly.

# 8. Troubleshooting

## `mcp: command not found`

Activate the virtual environment, then reinstall:

```bash
source .venv/bin/activate
python -m pip install "mcp[cli]>=1.27,<2"
```

On Windows PowerShell:

```powershell
.venv\Scripts\Activate.ps1
```

## `Connection closed`

Possible causes:

- syntax error in `server.py`;
- import error;
- server printed non-protocol data to stdout;
- wrong file path;
- incompatible SDK major version.

Check with:

```bash
python -m py_compile server.py client.py
mcp run server.py
```

## Resource is missing from `list_resources()`

A templated resource belongs to `list_resource_templates()`. This is
expected for `greeting://{name}`.

## Type mismatch

MCP validates arguments against the schema. These are valid integers:

```python
{"a": 1, "b": 7}
```

Strings such as `"one"` may fail validation.

# 9. Security and engineering notes

- Treat tool inputs as untrusted data.
- Add authorization before exposing sensitive tools.
- Restrict filesystem and subprocess access.
- Use timeouts for child processes and external calls.
- Avoid secrets in resource responses or logs.
- Keep STDIO stdout reserved for MCP protocol traffic.
- Pin SDK versions and run regression tests before upgrading.

# Deliverables checklist

- [x] MCP server named `Demo`
- [x] `add(a: int, b: int) -> int`
- [x] `greeting://{name}` resource template
- [x] STDIO server loop
- [x] Python STDIO client
- [x] Session initialization
- [x] Resource discovery
- [x] Resource-template discovery
- [x] Tool discovery
- [x] `greeting://hello` read
- [x] `add(1, 7)` call
- [x] Syntax checks
- [x] End-to-end integration smoke test
- [x] Terminal text capture
- [x] Thorough comments and learning documentation

# Conclusion

This exercise demonstrates the smallest useful local MCP architecture:

```text
Python client
    ↓ STDIO + MCP
Python server
    ├── Resource: greeting://{name}
    └── Tool: add(a, b)
```

MCP separates discovery and invocation from the implementation language.
A compatible client can understand the server's capabilities without
hard-coding the Python function internals.

# References

- Official MCP Python SDK, stable v1 branch:
  https://github.com/modelcontextprotocol/python-sdk/tree/v1.x
- MCP server guide:
  https://modelcontextprotocol.io/docs/develop/build-server
- MCP architecture:
  https://modelcontextprotocol.io/docs/learn/architecture